# 06 — Fused Attention

**Ladder:** Full $QK^{\top}$ (naive) → blocked tiles + **online softmax** → Triton fused forward (never store $N\times N$).

**Study tip.** Memorize the online-softmax rescale when the running max increases. Naive $S$ is $O(N^2)$ memory — that is the bug fusion fixes. Derive the $1/\sqrt D$ scale from the variance of a dot product.

Official: [Fused Attention](https://triton-lang.org/main/getting-started/tutorials/06-fused-attention.html)

Figures: [`docs/images/`](docs/images/) · guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

# Part I — Mathematics

## 0. Scaled dot-product attention

For one head, let $Q,K,V\in\mathbb{R}^{N\times D}$ with rows $q_i,k_i,v_i$. Define

$$
S=\alpha\,QK^{\top}\in\mathbb{R}^{N\times N},
\qquad
P=\mathrm{softmax}_{\mathrm{row}}(S),
\qquad
O=PV\in\mathbb{R}^{N\times D},
\tag{0.1}
$$

where

$$
S_{ij}=\alpha\,\langle q_i,k_j\rangle
=
\alpha\sum_{d=0}^{D-1}Q_{id}K_{jd},
\qquad
\alpha=\frac{1}{\sqrt D}.
\tag{0.2}
$$

Componentwise,

$$
P_{ij}
=
\frac{e^{S_{ij}}}{\sum_{\ell=0}^{N-1}e^{S_{i\ell}}},
\qquad
O_{id}
=
\sum_{j=0}^{N-1}P_{ij}V_{jd}.
\tag{0.3}
$$

Each output row is a convex combination of the value rows, because
$P_{i,:}\in\Delta^{N-1}$.

### 0.1 Why the $1/\sqrt D$ scale?

**Proposition 1 (variance of a dot product).** Suppose the entries of $q$ and
$k$ are independent with mean $0$ and variance $\sigma^{2}$. Then

$$
\mathbb{E}\big[\langle q,k\rangle\big]=0,
\qquad
\operatorname{Var}\big(\langle q,k\rangle\big)
=
D\,\sigma^{4}.
\tag{0.4}
$$

*Proof.* Each term $q_dk_d$ has mean $0$ and variance
$\mathbb{E}[q_d^{2}]\mathbb{E}[k_d^{2}]=\sigma^{4}$. The $D$ terms are
independent, so variances add. $\square$

Thus $\operatorname{sd}(\langle q,k\rangle)=\sigma^{2}\sqrt D$. Choosing
$\alpha=1/\sqrt D$ gives

$$
\operatorname{sd}(S_{ij})=\sigma^{2},
\tag{0.5}
$$

independent of the head dimension. Without the scale, logits would grow like
$\sqrt D$; for large $D$ the softmax saturates (one probability near $1$), its
Jacobian becomes ill-conditioned, and gradients vanish. The scale is therefore a
**conditioning** device, not a normalization of the data.

**Proposition 2 (row shift invariance).** For any row-dependent constants
$c_i$,

$$
\mathrm{softmax}_{\mathrm{row}}\!\left(S+c\mathbf{1}^{\top}\right)
=
\mathrm{softmax}_{\mathrm{row}}(S).
$$

*Proof.* Apply Proposition 2 of notebook 02 row by row. $\square$

### 0.2 Softmax row Jacobian

For a fixed row $i$, let $p=P_{i,:}\in\mathbb{R}^{N}$. Then

$$
\frac{\partial p_j}{\partial S_{i\ell}}
=
p_j\big(\delta_{j\ell}-p_\ell\big),
\qquad
J_i=\operatorname{diag}(p)-pp^{\top}.
\tag{0.6}
$$

The attention output is linear in $P$:

$$
O_{i,:}=p^{\top}V,
\qquad
\frac{\partial O_{i,d}}{\partial p_j}=V_{j,d}.
\tag{0.7}
$$

Combining (0.6) and (0.7), the gradient of the output with respect to the
logits is

$$
\frac{\partial O_{i,d}}{\partial S_{i,\ell}}
=
\sum_j V_{j,d}\,p_j(\delta_{j\ell}-p_\ell)
=
p_\ell\big(V_{\ell,d}-O_{i,d}\big).
\tag{0.8}
$$

This is the softmax-backward identity used by every attention backward kernel:
the gradient is a **centered** value, weighted by the probability. It is the
same projection structure as layer norm (4.19).

### 0.3 Complexity

**FLOPs.** $QK^{\top}$ costs $2N^{2}D$ and $PV$ costs $2N^{2}D$, so

$$
F=4N^{2}D
\qquad\text{(plus }O(N^{2})\text{ for softmax)}.
\tag{0.9}
$$

**Memory.** Materializing $S$ or $P$ costs $\Theta(N^{2})$ elements. For
$N=8192$ and `float16`, one such matrix is $128$ MiB — per head. The fused
algorithm stores only $O(ND)$ and $O(N)$ statistics, which is the entire point
of FlashAttention.

**Arithmetic intensity (naive).** Moving $Q,K,V,O$ is $O(ND)$ elements while
computing $O(N^{2}D)$ FLOPs, so the ideal intensity is $\Theta(N)$ FLOP/byte —
attention is compute-bound for long sequences **if** the $N^{2}$ intermediates
stay on chip. The naive implementation writes them to HBM, which turns the
problem memory-bound and dominates the runtime. Fusion restores the compute
bound.

### 0.4 Multi-head and batching

With $H$ heads of dimension $D_h=D/H$, the tensors have shape
$[Z,H,N,D_h]$. Each head is an independent attention problem; the heads are
concatenated along the feature axis and projected. Therefore

$$
F_{\mathrm{total}}=Z\,H\cdot4N^{2}D_h=4Z N^{2}D,
\tag{0.10}
$$

i.e. the same FLOPs as one head of dimension $D$, and the per-head algebra is
unchanged. This is why the kernel can put the head index in `program_id(1)`.

### 0.5 Worked example

Let $D=2$, $q=(1,0)$, $k_1=(1,0)$, $k_2=(0,1)$, $v_1=(1,0)$, $v_2=(0,1)$,
and $\alpha=1/\sqrt2$. Then

$$
\langle q,k_1\rangle=1,
\qquad
\langle q,k_2\rangle=0,
\qquad
S=\tfrac{1}{\sqrt2}(1,0).
$$

Hence

$$
P=\mathrm{softmax}\big((0.7071,0)\big)\approx(0.6698,0.3302),
\qquad
O\approx0.6698\,v_1+0.3302\,v_2=(0.6698,0.3302).
$$

**Teaching sizes.** $N=128$, $D=64$, `BLOCK_M=BLOCK_N=32` — four $Q$-tiles,
each streaming four $K/V$ tiles.

![Whiteboard: full $QK^\top$ vs blocked online-softmax](docs/images/benchmark-attention.png)

*Theory board — $O(N^2)$ score matrix vs online-softmax tiles that never store full $S$.*

### Theory board

![Whiteboard: full $QK^\top$ vs blocked online-softmax](docs/images/whiteboard-theory.jpg)

*Theory board — $O(N^2)$ score matrix vs online-softmax tiles that never store full $S$.*

# Part II — Proof

**Theorem (online softmax).** Let key blocks $B_1,\dots,B_T$ partition $[N]$,
and define $m^{(t)},\ell^{(t)},\mathrm{Acc}^{(t)}$ by (1.2). Then

$$
\ell^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{s_j-m^{(t)}},
\qquad
\mathrm{Acc}^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{s_j-m^{(t)}}v_j .
$$

*Proof by induction.* Base $t=0$ is empty. For the step, multiply the old sums
by $e^{m^{(t-1)}-m^{(t)}}$, which changes every exponent
$s_j-m^{(t-1)}$ into $s_j-m^{(t)}$; then add the new block. $\square$

**Corollary (attention equals softmax·V).**

$$
\frac{\mathrm{Acc}^{(T)}}{\ell^{(T)}}
=
\sum_j\mathrm{softmax}(s)_jv_j .
$$

*Proof.* Divide numerator and denominator by $e^{-m^{(T)}}$; the common factor
cancels. $\square$

**Proposition (two-pass equivalence).** A kernel that first computes
$(m,\ell)$ over all key blocks and then computes
$\sum_j e^{s_j-m}v_j$ produces the same output as the online recurrence.

*Proof.* Both compute the same quantities of the theorem with the same final
$m=m^{(T)}$. $\square$

# Part III — Step-by-step proof

1. **Load the query tile.** Read $Q_{\mathrm{tile}}\in\mathbb{R}^{B_M\times D}$
   once.
2. **Initialise.** $m=-\infty$, $\ell=0$, $\mathrm{Acc}=0$.
3. **Score a key tile.** Compute $s=\alpha Q_{\mathrm{tile}}K_t^{\top}$.
4. **Mask padding/future keys.** Set masked entries to $-\infty$.
5. **Update the max.** $m_{\mathrm{new}}=\max(m,\operatorname{rowmax}s)$.
6. **Rescale.** $\alpha_{\mathrm{resc}}=e^{m-m_{\mathrm{new}}}\le1$.
7. **Update mass and accumulator.**
   $\ell\leftarrow\alpha_{\mathrm{resc}}\ell+\operatorname{rowsum}e^{s-m_{\mathrm{new}}}$,
   $\mathrm{Acc}\leftarrow\alpha_{\mathrm{resc}}\mathrm{Acc}+e^{s-m_{\mathrm{new}}}V_t$.
8. **Repeat** for every key tile; the induction proves correctness.
9. **Finalize.** $O=\mathrm{Acc}/\ell$.
10. **Alternative.** The two-pass kernel performs steps 3–6 in a first loop and
    step 7 with the final $m$ in a second loop; it reads $K$ twice but never
    stores $S$.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import math

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (full S)

$$
S=\alpha QK^{\top},
\qquad
P=\mathrm{softmax}_{\mathrm{row}}(S),
\qquad
O=PV.
\tag{2.1}
$$

Implemented by `naive_attention_pseudo`. Clear reference; allocates
$\Theta(N^{2})$.

### 2.1 Cost of materialization

With element size $b$, the naive chain moves

$$
\underbrace{ND}_{\text{read }Q}
+\underbrace{ND}_{\text{read }K}
+\underbrace{N^{2}}_{\text{write }S}
+\underbrace{N^{2}}_{\text{read }S}
+\underbrace{N^{2}}_{\text{write }P}
+\underbrace{N^{2}}_{\text{read }P}
+\underbrace{ND}_{\text{read }V}
+\underbrace{ND}_{\text{write }O}
\ \text{elements}.
\tag{2.2}
$$

The $N^{2}$ terms dominate. In bytes,

$$
Q_{\mathrm{naive}}
\approx
4N^{2}b+O(NDb).
\tag{2.3}
$$

For $N=4096$, $b=2$: $4N^{2}b=128$ MiB of traffic **per head**, versus
$O(NDb)=1$ MiB for the inputs and outputs. This $128:1$ ratio is why naive
attention is bandwidth-bound even though it performs $4N^{2}D$ FLOPs.

**Memory.** The two $N\times N$ tensors $S$ and $P$ must coexist (or be fused
away); at $N=4096$ each is $32$ MiB in FP16. Long-context models would need
gigabytes.

### 2.2 Why keep it?

The naive version is the mathematical definition (0.1). It is the reference
against which the blocked and Triton versions are tested
(`test_pseudo_matches_torch`). Its role in the notebook is correctness, not
performance.


In [ ]:
def naive_attention_pseudo(Q, K, V):
    """
    Standard scaled-dot-product attention for one (or batched) head:
    (See markdown cells above for the math.)
    """
    D = Q.shape[-1]
    scale = 1.0 / math.sqrt(D)

    # Full QK^T — allocates an N×N tensor (bad for long sequences)
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale
    P = torch.softmax(S, dim=-1)
    O = torch.matmul(P, V)
    return O


### Method 02 — Kernel-based (two-pass, no online fusion)

This kernel assigns a tile of query rows to each program and computes the
softmax in **two passes over the keys**: pass 1 obtains the row maximum and
normalizer, pass 2 accumulates $P V$ using that final maximum. It is a real
Triton kernel and never stores $S$, but it reads $K$ twice — the non-fused
baseline for Method 03.


In [ ]:
@triton.jit
def _attn_fwd_kernel_twopass(
    Q,
    K,
    V,
    Out,
    stride_qz,
    stride_qh,
    stride_qm,
    stride_qk,
    stride_kz,
    stride_kh,
    stride_kn,
    stride_kk,
    stride_vz,
    stride_vh,
    stride_vn,
    stride_vk,
    stride_oz,
    stride_oh,
    stride_om,
    stride_ok,
    Z,
    H,
    N_CTX,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_DMODEL: tl.constexpr,
):
    """Method 02: two passes over K — final max first, then accumulate P V."""
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H
    q_offset = off_z.to(tl.int64) * stride_qz + off_h.to(tl.int64) * stride_qh
    k_offset = off_z.to(tl.int64) * stride_kz + off_h.to(tl.int64) * stride_kh
    v_offset = off_z.to(tl.int64) * stride_vz + off_h.to(tl.int64) * stride_vh
    o_offset = off_z.to(tl.int64) * stride_oz + off_h.to(tl.int64) * stride_oh

    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_DMODEL)
    q_ptrs = (
        Q
        + q_offset
        + offs_m[:, None] * stride_qm
        + offs_d[None, :] * stride_qk
    )
    q_mask = offs_m[:, None] < N_CTX
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    scale = 1.0 / tl.sqrt(tl.cast(BLOCK_DMODEL, tl.float32))
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)

    # Pass 1: running max and normalizer.
    for start_n in range(0, N_CTX, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)
        k_ptrs = (
            K
            + k_offset
            + offs_n[:, None] * stride_kn
            + offs_d[None, :] * stride_kk
        )
        k_mask = offs_n[:, None] < N_CTX
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)
        qk = tl.dot(q, tl.trans(k)) * scale
        qk = tl.where(offs_n[None, :] < N_CTX, qk, float("-inf"))

        m_ij = tl.max(qk, axis=1)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new[:, None])
        l_i = l_i * alpha + tl.sum(p, axis=1)
        m_i = m_new

    # Pass 2: accumulate with the final maximum (no more rescaling).
    acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)
    for start_n in range(0, N_CTX, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)
        k_ptrs = (
            K
            + k_offset
            + offs_n[:, None] * stride_kn
            + offs_d[None, :] * stride_kk
        )
        k_mask = offs_n[:, None] < N_CTX
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)
        qk = tl.dot(q, tl.trans(k)) * scale
        qk = tl.where(offs_n[None, :] < N_CTX, qk, float("-inf"))
        p = tl.exp(qk - m_i[:, None])

        v_ptrs = (
            V
            + v_offset
            + offs_n[:, None] * stride_vn
            + offs_d[None, :] * stride_vk
        )
        v = tl.load(v_ptrs, mask=k_mask, other=0.0)
        acc += tl.dot(p.to(v.dtype), v)

    acc = acc / l_i[:, None]
    o_ptrs = (
        Out
        + q_offset
        + offs_m[:, None] * stride_om
        + offs_d[None, :] * stride_ok
    )
    tl.store(o_ptrs, acc.to(Out.dtype.element_ty), mask=q_mask)


def attention_twopass(q, k, v, BLOCK_M=64, BLOCK_N=64):
    """Method 02 launcher: two-pass (non-online) fused-memory attention."""
    Z, H, N_CTX, D = q.shape
    assert k.shape == v.shape == (Z, H, N_CTX, D)
    assert D == triton.next_power_of_2(D), (
        "this teaching kernel uses BLOCK_DMODEL = D; choose a power-of-two head dim"
    )
    out = torch.empty_like(q)
    grid = (triton.cdiv(N_CTX, BLOCK_M), Z * H)
    _attn_fwd_kernel_twopass[grid](
        q,
        k,
        v,
        out,
        q.stride(0),
        q.stride(1),
        q.stride(2),
        q.stride(3),
        k.stride(0),
        k.stride(1),
        k.stride(2),
        k.stride(3),
        v.stride(0),
        v.stride(1),
        v.stride(2),
        v.stride(3),
        out.stride(0),
        out.stride(1),
        out.stride(2),
        out.stride(3),
        Z,
        H,
        N_CTX,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_DMODEL=D,
    )
    return out


### Method 03 — Fused kernel-based (online softmax)

Process key blocks $t=1,2,\dots,T$ with scores $s^{(t)}\in\mathbb{R}^{B_N}$
(for a fixed query row) and values $V^{(t)}\in\mathbb{R}^{B_N\times D}$.
Maintain

$$
m^{(t)}\in\mathbb{R},
\qquad
\ell^{(t)}\in\mathbb{R},
\qquad
\mathrm{Acc}^{(t)}\in\mathbb{R}^{D}.
\tag{1.1}
$$

**Initialize:** $m^{(0)}=-\infty$, $\ell^{(0)}=0$,
$\mathrm{Acc}^{(0)}=0$.

**Update (row-wise):**

$$
\begin{aligned}
m^{(t)}
&=
\max\!\left(m^{(t-1)},\ \max_j s^{(t)}_j\right),\\[3pt]
\ell^{(t)}
&=
e^{\,m^{(t-1)}-m^{(t)}}\,\ell^{(t-1)}
+
\sum_j e^{\,s^{(t)}_j-m^{(t)}},\\[3pt]
\mathrm{Acc}^{(t)}
&=
e^{\,m^{(t-1)}-m^{(t)}}\,\mathrm{Acc}^{(t-1)}
+
\sum_j e^{\,s^{(t)}_j-m^{(t)}}\,v^{(t)}_j .
\end{aligned}
\tag{1.2}
$$

**Finalize:** after the last block $T$,

$$
O_{\mathrm{tile}}=\frac{\mathrm{Acc}^{(T)}}{\ell^{(T)}}.
\tag{1.3}
$$

**Theorem 1 (exactness).** For every $t\ge0$,

$$
m^{(t)}=\max_{u\le t}\max_{j\in B_u}s_j,
\qquad
\ell^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{\,s_j-m^{(t)}},
\qquad
\mathrm{Acc}^{(t)}=\sum_{u\le t}\sum_{j\in B_u}e^{\,s_j-m^{(t)}}v_j .
\tag{1.4}
$$

*Proof by induction.* Base $t=0$ is the empty sum. Let $\mu=m^{(t)}$. The first
identity is the definition of the running max. For the second,

$$
\begin{aligned}
\ell^{(t)}
&=
e^{m^{(t-1)}-\mu}
\sum_{u\le t-1}\sum_{j\in B_u}e^{s_j-m^{(t-1)}}
+
\sum_{j\in B_t}e^{s_j-\mu}\\
&=
\sum_{u\le t-1}\sum_{j\in B_u}e^{s_j-\mu}
+
\sum_{j\in B_t}e^{s_j-\mu}
=
\sum_{u\le t}\sum_{j\in B_u}e^{s_j-\mu}.
\end{aligned}
$$

Appending the factor $v_j$ to every term proves the third identity. $\square$

**Corollary 2 (final attention).**

$$
O_{\mathrm{tile}}
=
\frac{\sum_{j\in[N]}e^{s_j-m^{(T)}}v_j}
{\sum_{j\in[N]}e^{s_j-m^{(T)}}}
=
\sum_{j\in[N]}\mathrm{softmax}(s)_j\,v_j .
\tag{1.5}
$$

*Proof.* Multiply numerator and denominator by $e^{-m^{(T)}}$; the common
factor cancels. $\square$

### 1.1 The rescale factor

Define

$$
\alpha^{(t)}:=e^{\,m^{(t-1)}-m^{(t)}}.
\tag{1.6}
$$

Because $m^{(t)}\ge m^{(t-1)}$, we have $0<\alpha^{(t)}\le1$. It can only
**shrink** the old accumulator, never amplify it, so no overflow is introduced
by the rescale. On the first block $m^{(0)}=-\infty$, hence
$\alpha^{(1)}=e^{-\infty}=0$ and the old (empty) state is discarded.

**Implementation detail.** The code computes `alpha = exp(m_i - m_new)` after
`m_new = maximum(m_i, m_ij)`. In exact arithmetic the exponent is $\le0$. In
floating point, if `m_i == m_new == -inf` (a fully masked first block),
`-inf - (-inf)` is NaN; production kernels guard this by initializing with a
finite large negative value or by special-casing the first iteration. The
teaching kernel avoids the issue because the first block always has at least
one valid key.

### 1.2 Block scores

For a query tile $Q_{\mathrm{tile}}\in\mathbb{R}^{B_M\times D}$ and key tile
$K^{(t)}\in\mathbb{R}^{B_N\times D}$,

$$
s^{(t)}=\alpha\,Q_{\mathrm{tile}}\big(K^{(t)}\big)^{\top}
\in\mathbb{R}^{B_M\times B_N},
\qquad
v^{(t)}=V^{(t)}\in\mathbb{R}^{B_N\times D}.
\tag{1.7}
$$

All updates (1.2) are row-wise in the $B_M$ query rows; the $D$ columns of
$\mathrm{Acc}$ are independent. In matrix form,

$$
\begin{aligned}
m^{(t)}&=\max\!\left(m^{(t-1)},\ \operatorname{rowmax}s^{(t)}\right),\\
\ell^{(t)}&=\alpha^{(t)}\ell^{(t-1)}+\operatorname{rowsum}\!\left(e^{s^{(t)}-m^{(t)}}\right),\\
\mathrm{Acc}^{(t)}&=\alpha^{(t)}\mathrm{Acc}^{(t-1)}
+\left(e^{s^{(t)}-m^{(t)}}\right)V^{(t)},
\end{aligned}
\tag{1.8}
$$

where $\alpha^{(t)}$ is broadcast over rows and the exponent is applied
elementwise.

### 1.3 Numerical stability

Each exponent satisfies $s^{(t)}_j-m^{(t)}\le0$, so all exponentials lie in
$(0,1]$ and the sum $\ell^{(t)}$ lies in $(0,N]$. The accumulator is a weighted
sum of values with weights in $[0,1]$; its magnitude is bounded by
$\max_j|v_j|\,\ell^{(t)}$, so it cannot overflow before the final division.
This is the same max-subtraction trick as notebook 02, applied incrementally.

### 1.4 Causal masking

For causal attention, $S_{ij}=-\infty$ whenever $j>i$. In block form, a key
tile $t$ contributes only to query rows $i\ge j$ for $j\in B_t$. Two
optimizations follow:

1. **Masking.** Set $s^{(t)}_{ij}=-\infty$ for masked positions before the max;
   then $e^{-\infty}=0$ and they contribute nothing.
2. **Skipping.** If the entire key tile is strictly in the future of the query
   tile, skip it entirely. For a lower-triangular mask this halves the number of
   processed tiles:

$$
\#\text{tiles}_{\mathrm{causal}}
\approx
\frac12\left(\frac{N}{B_M}\right)\left(\frac{N}{B_N}\right)
+
O\!\left(\frac{N}{B_M}\right).
\tag{1.9}
$$

Hence causal attention costs about half the FLOPs and half the memory traffic
of full attention.

### 1.5 What is stored?

Only $(m,\ell,\mathrm{Acc})$ per query row, i.e. $O(B_MD)$ state for a tile,
independent of $N$. The full $S$ and $P$ matrices are never materialized. This
is the $O(N)$-memory property.


#### Python model of the online recurrence

For each $Q$-tile of height $B_M$, loop over $K/V$-tiles of height $B_N$:

$$
s^{(t)}=\alpha\,Q_{\mathrm{tile}}\big(K^{(t)}\big)^{\top},
\quad\text{then apply the §1 updates with }V^{(t)}.
\tag{3.1}
$$

Implemented by `blocked_attention_pseudo`.

### 3.1 Scratch memory

For one query tile the algorithm keeps

$$
\underbrace{B_M\times B_N}_{s^{(t)}}
+
\underbrace{B_M\times D}_{\mathrm{Acc}}
+
\underbrace{B_M}_{\ell}
+
\underbrace{B_M}_{m}
\tag{3.2}
$$

elements. The $K/V$ tiles are streamed and need only $B_N\times D$ each.
Therefore

$$
\text{scratch}
=
O(B_MB_N+B_MD+B_ND),
\tag{3.3}
$$

**independent of $N$**. With $B_M=B_N=32$ and $D=64$, this is at most a few
thousand elements — small enough for SRAM.

### 3.2 FLOPs and traffic

The FLOPs are unchanged:

$$
F=4N^{2}D,
\tag{3.4}
$$

because every score is still computed and every score multiplies a value. The
savings are in **memory**, not arithmetic:

$$
Q_{\mathrm{blocked}}
=
\underbrace{ND}_{\text{read }Q}
+
\underbrace{ND}_{\text{write }O}
+
\text{streamed }K,V\text{ tiles}.
\tag{3.5}
$$

Each $Q$-tile streams all $K,V$, so $K$ and $V$ are read $N/B_M$ times:

$$
Q_{\mathrm{blocked}}
\approx
ND+ND+
2\,\frac{N}{B_M}\,ND
=
O\!\left(\frac{N^{2}D}{B_M}\right)
\ \text{elements}.
\tag{3.6}
$$

For $B_M\approx D$ this is $O(N^{2})$ elements, still far below the $4N^{2}$
**extra** traffic of the naive version, and — crucially — the peak memory is
$O(ND)$ instead of $O(N^{2})$.

### 3.3 Correctness

The blocked version computes exactly the same $O$ as the naive version because
Theorem 1 shows that the online recurrence is algebraically identical to a
single softmax over the full row. The test compares the two with
`atol=1e-4`; differences are only floating-point rounding.

| Stage | Function | Stores full $S$? | Peak memory |
|-------|----------|------------------|-------------|
| Naive | `naive_attention_pseudo` | yes | $O(N^{2})$ |
| Blocked online | `blocked_attention_pseudo` | no | $O(ND+B_MB_N)$ |
| Triton | `_attn_fwd_kernel` / `attention` | no | $O(ND)$ |


In [ ]:
def blocked_attention_pseudo(Q, K, V, BLOCK_M=32, BLOCK_N=32):
    """
    Same math as naive, but process query rows in BLOCK_M tiles and key/value
    (See markdown cells above for the math.)
    """
    *batch, N, D = Q.shape
    assert N % BLOCK_M == 0 and N % BLOCK_N == 0, "assume divisible for teaching"

    scale = 1.0 / math.sqrt(D)
    O = torch.zeros_like(Q, memory_format=torch.contiguous_format)

    # Flatten leading batch dims so we can loop over heads/batches simply
    Qf = Q.reshape(-1, N, D)
    Kf = K.reshape(-1, N, D)
    Vf = V.reshape(-1, N, D)
    Of = O.reshape(-1, N, D)

    for b in range(Qf.shape[0]):
        for start_m in range(0, N, BLOCK_M):
            q = Qf[b, start_m : start_m + BLOCK_M, :]  # [BLOCK_M, D]
            # Accumulate attention over key blocks with online softmax
            m_i = torch.full((BLOCK_M,), -float("inf"), dtype=torch.float32, device=Q.device)
            l_i = torch.zeros(BLOCK_M, dtype=torch.float32, device=Q.device)
            acc = torch.zeros(BLOCK_M, D, dtype=torch.float32, device=Q.device)

            for start_n in range(0, N, BLOCK_N):
                k = Kf[b, start_n : start_n + BLOCK_N, :]  # [BLOCK_N, D]
                v = Vf[b, start_n : start_n + BLOCK_N, :]  # [BLOCK_N, D]
                qk = (q.float() @ k.float().T) * scale  # [BLOCK_M, BLOCK_N]

                m_ij = qk.max(dim=-1).values
                m_new = torch.maximum(m_i, m_ij)
                alpha = torch.exp(m_i - m_new)
                p = torch.exp(qk - m_new[:, None])
                l_i = l_i * alpha + p.sum(dim=-1)
                m_i = m_new
                acc = acc * alpha[:, None] + p @ v.float()

            Of[b, start_m : start_m + BLOCK_M, :] = (acc / l_i[:, None]).to(Q.dtype)

    return O


#### Triton fused kernel

Same online-softmax algebra in SRAM: load $Q$/$K$/$V$ tiles, use `tl.dot` for
scores, update $(m,\ell,\mathrm{Acc})$, store $O$ only.

### 4.1 Program mapping

The grid is

$$
\mathrm{grid}
=
\left(
\left\lceil\frac{N}{B_M}\right\rceil,\ Z\cdot H
\right).
\tag{4.1}
$$

Program $(p,zh)$ owns query rows
$pB_M+\{0,\dots,B_M-1\}$ for batch/head pair
$z=\lfloor zh/H\rfloor$, $h=zh\bmod H$. The pointer offset into the
$[Z,H,N,D]$ tensors is

$$
\mathrm{offset}=z\,s_z+h\,s_h
\tag{4.2}
$$

and each tile adds $m\,s_m+d\,s_d$. This is the standard stride arithmetic; the
kernel never assumes contiguity across heads.

### 4.2 The loop body

For each key tile $n=0,B_N,2B_N,\dots$:

$$
\begin{aligned}
qk &= \alpha\,Q_{\mathrm{tile}}K_n^{\top},\\
qk_{:,\,j} &\leftarrow-\infty &&\text{if }j\ge N\ \text{(padding)},\\
m_{\mathrm{new}}&=\max(m_i,\operatorname{rowmax}qk),\\
\alpha_{\mathrm{resc}}&=e^{m_i-m_{\mathrm{new}}},\\
p&=e^{qk-m_{\mathrm{new}}},\\
\ell_i&\leftarrow\alpha_{\mathrm{resc}}\ell_i+\operatorname{rowsum}p,\\
\mathrm{Acc}&\leftarrow\alpha_{\mathrm{resc}}\mathrm{Acc}+p\,V_n .
\end{aligned}
\tag{4.3}
$$

After the loop,

$$
O_{\mathrm{tile}}=\mathrm{Acc}/\ell_i .
\tag{4.4}
$$

The division is deferred to the end, so the loop works with unnormalized
weights; this is essential because the normalization factor changes as $m$
grows.

### 4.3 Why `tl.dot` is safe here

`tl.dot` on FP16 inputs with an FP32 accumulator computes the $B_M\times B_N$
score tile at tensor-core speed. The scores are immediately consumed by the
softmax update, so the $N^{2}$ matrix never exists in global memory. The
accumulator `acc` is FP32 to preserve precision across $N/B_N$ updates.

### 4.4 Cost analysis

- **FLOPs:** $4N^{2}D$ — unchanged.
- **HBM traffic:** $Q$ read once ($ND$), $O$ written once ($ND$), $K,V$ read
  once per $Q$-tile $\Rightarrow O(N^{2}D/B_M)$ element reads.
- **Peak memory:** $O(ND+B_MB_N+B_MD)$ — independent of $N^{2}$.

The speedup over naive attention is therefore **I/O**, not FLOPs: both do the
same arithmetic, but the fused kernel avoids writing and re-reading $S$ and $P$.
For long $N$ this can be several times faster even on the same hardware.

**Optional causal mask.** Set $S_{ij}=-\infty$ for $j>i$ (or skip pure-future
$K$-tiles). This halves both FLOPs and traffic (1.9). The kernel in this
notebook implements the non-causal path; adding causal support requires only the
mask and a loop bound `start_n <= (start_m+1)*BLOCK_M`.

### 4.5 Numerical error

The computed output satisfies

$$
\big|\hat O_{id}-O_{id}\big|
\le
C\,u\,N\,\max_j|V_{jd}|,
\tag{4.5}
$$

where $u$ is the unit roundoff of the accumulator and $C$ is a small constant.
The factor $N$ comes from summing $N$ softmax weights. With FP16 inputs and
FP32 accumulation, $u=2^{-24}$ and the error is dominated by the input
quantization ($2^{-11}$), which is why the test tolerance is $2\times10^{-2}$.

### 4.6 Summary

| Property | Naive | Fused |
|---|---|---|
| FLOPs | $4N^{2}D$ | $4N^{2}D$ |
| Peak memory | $O(N^{2})$ | $O(ND)$ |
| $S/P$ in HBM | yes | no |
| Parallelism | matrix kernels | query tiles $\times$ heads |
| Bottleneck | bandwidth (HBM) | compute / SRAM |

The fused kernel realizes the same bilinear-plus-softmax map as (0.1); only the
schedule and the storage of intermediates change.


In [ ]:
@triton.jit
def _attn_fwd_kernel(
    Q,
    K,
    V,
    Out,
    stride_qz,
    stride_qh,
    stride_qm,
    stride_qk,
    stride_kz,
    stride_kh,
    stride_kn,
    stride_kk,
    stride_vz,
    stride_vh,
    stride_vn,
    stride_vk,
    stride_oz,
    stride_oh,
    stride_om,
    stride_ok,
    Z,
    H,
    N_CTX,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_DMODEL: tl.constexpr,
):
    """One program = one query tile of BLOCK_M rows for one (batch, head)."""
    start_m = tl.program_id(0)
    off_hz = tl.program_id(1)
    off_z = off_hz // H
    off_h = off_hz % H

    q_offset = off_z.to(tl.int64) * stride_qz + off_h.to(tl.int64) * stride_qh
    k_offset = off_z.to(tl.int64) * stride_kz + off_h.to(tl.int64) * stride_kh
    v_offset = off_z.to(tl.int64) * stride_vz + off_h.to(tl.int64) * stride_vh
    o_offset = off_z.to(tl.int64) * stride_oz + off_h.to(tl.int64) * stride_oh

    # Load Q tile once; stream K/V tiles along the sequence
    offs_m = start_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_DMODEL)
    q_ptrs = Q + q_offset + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qk
    q_mask = offs_m[:, None] < N_CTX
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    acc = tl.zeros([BLOCK_M, BLOCK_DMODEL], dtype=tl.float32)
    m_i = tl.zeros([BLOCK_M], dtype=tl.float32) - float("inf")
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    scale = 1.0 / tl.sqrt(tl.cast(BLOCK_DMODEL, tl.float32))

    for start_n in range(0, N_CTX, BLOCK_N):
        start_n = tl.multiple_of(start_n, BLOCK_N)
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = K + k_offset + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kk
        k_mask = offs_n[:, None] < N_CTX
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)

        qk = tl.dot(q, tl.trans(k)) * scale
        # Mask padded key positions so they never win the max / get weight
        qk = tl.where(offs_n[None, :] < N_CTX, qk, float("-inf"))

        m_ij = tl.max(qk, axis=1)
        m_new = tl.maximum(m_i, m_ij)
        alpha = tl.exp(m_i - m_new)
        p = tl.exp(qk - m_new[:, None])
        l_i = l_i * alpha + tl.sum(p, axis=1)
        m_i = m_new

        v_ptrs = V + v_offset + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vk
        v = tl.load(v_ptrs, mask=k_mask, other=0.0)
        acc = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)

    acc = acc / l_i[:, None]
    o_ptrs = Out + o_offset + offs_m[:, None] * stride_om + offs_d[None, :] * stride_ok
    tl.store(o_ptrs, acc.to(Out.dtype.element_ty), mask=q_mask)


def attention(q, k, v, BLOCK_M=64, BLOCK_N=64):
    """Launch fused forward attention. q/k/v: [Z, H, N_CTX, D]."""
    Z, H, N_CTX, D = q.shape
    assert k.shape == v.shape == (Z, H, N_CTX, D)
    assert D == triton.next_power_of_2(D), (
        "this teaching kernel uses BLOCK_DMODEL = D; choose a power-of-two head dim"
    )
    out = torch.empty_like(q)
    grid = (triton.cdiv(N_CTX, BLOCK_M), Z * H)
    _attn_fwd_kernel[grid](
        q,
        k,
        v,
        out,
        q.stride(0),
        q.stride(1),
        q.stride(2),
        q.stride(3),
        k.stride(0),
        k.stride(1),
        k.stride(2),
        k.stride(3),
        v.stride(0),
        v.stride(1),
        v.stride(2),
        v.stride(3),
        out.stride(0),
        out.stride(1),
        out.stride(2),
        out.stride(3),
        Z,
        H,
        N_CTX,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_DMODEL=D,
    )
    return out


# Alias used in docs / older snippets
attention_ref = naive_attention_pseudo


### Pitfalls

- Materializing $S=QK^{\top}$ is $O(N^{2})$ memory — blows up for long context.
- Online softmax must rescale the previous accumulator when $m$ updates.
- Wrong scale $\alpha=1/\sqrt D$ changes attention sharpness.
- Padding keys must be masked to $-\infty$ before the max.

### Self-check

1. Derive (0.4) and explain why $\alpha=1/\sqrt D$.
2. State and prove the online update for $(m,\ell,\mathrm{Acc})$.
3. Derive the attention gradient (0.8) from the softmax Jacobian.
4. Compare memory: classic vs Flash-style.


In [ ]:
def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    Z, H, N, D = 1, 2, 64, 32
    q = torch.randn(Z, H, N, D)
    k = torch.randn(Z, H, N, D)
    v = torch.randn(Z, H, N, D)
    ref = naive_attention_pseudo(q, k, v)
    blocked = blocked_attention_pseudo(q, k, v, BLOCK_M=32, BLOCK_N=32)
    assert torch.allclose(blocked, ref, atol=1e-4, rtol=1e-4)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    Z, H, N_CTX, D = 1, 2, 128, 64
    q = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    k = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    v = torch.randn(Z, H, N_CTX, D, device=DEVICE, dtype=torch.float16)
    ref = naive_attention_pseudo(q.float(), k.float(), v.float()).half()
    out = attention(q, k, v, BLOCK_M=64, BLOCK_N=64)
    assert torch.allclose(out, ref, atol=2e-2, rtol=2e-2)
    print(f"fused attention OK  shape={tuple(out.shape)}")


In [ ]:
def test_kernel_methods_attention() -> None:
    """Method 01 (full S) vs Method 02 (two-pass) vs Method 03 (online)."""
    require_gpu()
    torch.manual_seed(0)
    Z, H, N, D = 1, 2, 128, 64
    q = torch.randn(Z, H, N, D, device=DEVICE, dtype=torch.float16)
    k = torch.randn(Z, H, N, D, device=DEVICE, dtype=torch.float16)
    v = torch.randn(Z, H, N, D, device=DEVICE, dtype=torch.float16)
    ref = naive_attention_pseudo(q.float(), k.float(), v.float()).half()
    assert torch.allclose(
        attention_twopass(q, k, v), ref, atol=2e-2, rtol=2e-2
    )
    assert torch.allclose(attention(q, k, v), ref, atol=2e-2, rtol=2e-2)
    print("attention: methods 02 and 03 OK")
